# Massa-mola: definicao de derivada, Euler e RK4

Este material compara tres metodos numericos para o sistema massa-mola sem amortecimento.
Neste ajuste didatico, focamos apenas em:
- grafico de posicao x(t);
- aparato experimental usado na animacao.
        


## Aparato experimental considerado

Modelo adotado:
- uma parede fixa;
- mola ideal ligada a uma massa que oscila em 1D;
- sem amortecimento e sem forca externa.

EDO:
`x'' + (k/m) * x = 0`
        


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams['animation.html'] = 'jshtml'
plt.rcParams['figure.figsize'] = (10, 4)
        


In [ ]:
# BLOCO 1 - Parametros
m = 1.0
k = 1.0
x0 = 1.0
v0 = 0.0
dt = 0.01
t_final = 20.0
t = np.arange(0.0, t_final + dt, dt)


def mostrar_amostra(nome, t, x, v, n=10):
    print(f"\nSaida numerica - {nome}")
    print(' i   t(s)      x(m)        v(m/s)')
    for i in range(min(n, len(t))):
        print(f"{i:2d}  {t[i]:6.3f}   {x[i]:10.6f}   {v[i]:10.6f}")
        


In [ ]:
# BLOCO 2 - Definicao de derivada (diferenca finita de 2a ordem)
def simular_definicao_derivada(t, m, k, x0, v0):
    dt = t[1] - t[0]
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)

    x[0] = x0
    a0 = -(k / m) * x0
    x[1] = x0 + v0 * dt + 0.5 * a0 * dt**2

    for i in range(1, n - 1):
        a_i = -(k / m) * x[i]
        x[i + 1] = 2 * x[i] - x[i - 1] + a_i * dt**2

    v[0] = v0
    v[1:-1] = (x[2:] - x[:-2]) / (2 * dt)
    v[-1] = (x[-1] - x[-2]) / dt
    return x, v


# BLOCO 3 - Euler
def simular_euler(t, m, k, x0, v0):
    dt = t[1] - t[0]
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)

    x[0] = x0
    v[0] = v0

    for i in range(n - 1):
        a = -(k / m) * x[i]
        v[i + 1] = v[i] + a * dt
        x[i + 1] = x[i] + v[i] * dt

    return x, v


# BLOCO 4 - RK4
def derivadas(x, v, m, k):
    return v, -(k / m) * x


def simular_rk4(t, m, k, x0, v0):
    dt = t[1] - t[0]
    n = len(t)
    x = np.zeros(n)
    v = np.zeros(n)

    x[0] = x0
    v[0] = v0

    for i in range(n - 1):
        xi, vi = x[i], v[i]

        k1_x, k1_v = derivadas(xi, vi, m, k)
        k2_x, k2_v = derivadas(xi + 0.5 * dt * k1_x, vi + 0.5 * dt * k1_v, m, k)
        k3_x, k3_v = derivadas(xi + 0.5 * dt * k2_x, vi + 0.5 * dt * k2_v, m, k)
        k4_x, k4_v = derivadas(xi + dt * k3_x, vi + dt * k3_v, m, k)

        x[i + 1] = xi + (dt / 6.0) * (k1_x + 2 * k2_x + 2 * k3_x + k4_x)
        v[i + 1] = vi + (dt / 6.0) * (k1_v + 2 * k2_v + 2 * k3_v + k4_v)

    return x, v
        


In [ ]:
# BLOCO 5 - Execucao
x_def, v_def = simular_definicao_derivada(t, m, k, x0, v0)
x_euler, v_euler = simular_euler(t, m, k, x0, v0)
x_rk4, v_rk4 = simular_rk4(t, m, k, x0, v0)

mostrar_amostra('Definicao de derivada', t, x_def, v_def)
mostrar_amostra('Euler', t, x_euler, v_euler)
mostrar_amostra('RK4', t, x_rk4, v_rk4)
        


In [ ]:
# BLOCO 6 - Grafico comparativo (somente posicao)
plt.figure(figsize=(10, 4))
plt.plot(t, x_def, label='Def. derivada', linewidth=1.2)
plt.plot(t, x_euler, label='Euler', linewidth=1.2)
plt.plot(t, x_rk4, label='RK4', linewidth=2)
plt.title('Comparacao de posicao x(t)')
plt.xlabel('tempo (s)')
plt.ylabel('x (m)')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
        


## Animacao do aparato experimental

A animacao abaixo mostra o aparato massa-mola e, no painel inferior, apenas a posicao x(t).
        


In [ ]:
def desenhar_mola(x_massa, x_parede=-1.2, espiras=12, amplitude=0.08, pontos=220):
    x_final = x_massa - 0.08
    if x_final <= x_parede + 0.05:
        x_final = x_parede + 0.05

    xs = np.linspace(x_parede, x_final, pontos)
    fase = np.linspace(0.0, 2.0 * np.pi * espiras, pontos)
    ys = amplitude * np.sin(fase)
    ys[0] = 0.0
    ys[-1] = 0.0
    return xs, ys


def animar_massa_mola(t, x, titulo='Massa-mola', max_frames=350):
    n = len(t)
    passo = max(1, n // max_frames)
    t_anim = t[::passo]
    x_anim = x[::passo]

    fig, (ax_mola, ax_xt) = plt.subplots(
        2,
        1,
        figsize=(8, 7),
        gridspec_kw={'height_ratios': [1.3, 1.0]},
    )

    ax_mola.set_title(f'{titulo} - Aparato massa-mola')
    ax_mola.set_xlabel('x (m)')
    ax_mola.set_ylabel('y')
    xmin = min(-1.3, np.min(x_anim) - 0.3)
    xmax = max(1.3, np.max(x_anim) + 0.3)
    ax_mola.set_xlim(xmin, xmax)
    ax_mola.set_ylim(-0.4, 0.4)
    ax_mola.grid(alpha=0.3)
    ax_mola.axvline(-1.2, color='gray', lw=3)

    linha_mola, = ax_mola.plot([], [], color='tab:blue', lw=2)
    massa, = ax_mola.plot([], [], 'o', color='tab:red', ms=14)

    ax_xt.set_title('Posicao x(t)')
    ax_xt.set_xlabel('tempo (s)')
    ax_xt.set_ylabel('x (m)')
    ax_xt.set_xlim(0.0, t[-1])
    margem = 0.1 * max(abs(np.min(x_anim)), abs(np.max(x_anim)), 1e-6)
    ax_xt.set_ylim(np.min(x_anim) - margem, np.max(x_anim) + margem)
    ax_xt.grid(alpha=0.3)
    linha_xt, = ax_xt.plot([], [], color='tab:orange', lw=2)

    def init():
        linha_mola.set_data([], [])
        massa.set_data([], [])
        linha_xt.set_data([], [])
        return linha_mola, massa, linha_xt

    def update(i):
        xs, ys = desenhar_mola(x_anim[i])
        linha_mola.set_data(xs, ys)
        massa.set_data([x_anim[i]], [0.0])
        linha_xt.set_data(t_anim[: i + 1], x_anim[: i + 1])
        return linha_mola, massa, linha_xt

    ani = FuncAnimation(fig, update, frames=len(x_anim), init_func=init, interval=30, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())


metodos = {
    'Definicao de derivada': x_def,
    'Euler': x_euler,
    'RK4': x_rk4,
}

metodo_animacao = 'RK4'  # Troque para 'Euler' ou 'Definicao de derivada'.
display(animar_massa_mola(t, metodos[metodo_animacao], titulo=f'Massa-mola - {metodo_animacao}'))
        
